# EXXA — Scaled Conditional DDPM (Week-4 plan, Kaggle T4×2)

Standalone notebook for the **official Week-4 deliverable**: a larger conditional Denoising Diffusion
Probabilistic Model, trained on Kaggle's **T4×2**.

| Item | This notebook |
|---|---|
| Backbone | `DiffusionUNet` ch=128, ch_mult=`[1,1,2,2,4,4]` (**6 levels**, attn@16) — **~110M params** |
| Resolution | **128×128** random crops |
| Conditioning | `cat([dirty, noisy_clean])` → predict ε |
| Diffusion | T=1000, linear (or cosine) β-schedule |
| Multi-GPU | `nn.DataParallel` across both T4s (automatic) |
| Sampling | DDIM, 50 steps |
| Deliverables | loss curve + first sample outputs + preliminary PSNR/SSIM/MSE |

### Before you run
1. **Accelerator = GPU T4 ×2** · **Internet = On**
2. *Add Input* → your `dirty.npy` / `clean.npy` Kaggle Dataset (auto-discovered under `/kaggle/input/`).


## 1. Setup — clone fork, install deps, verify both GPUs (Day 1.1)


In [ ]:
import os, sys, subprocess, glob

REPO_URL = 'https://github.com/KrishanYadav333/EXXA.git'
REPO_DIR = '/kaggle/working/EXXA'
PKG_ROOT = os.path.join(REPO_DIR, 'DENOISING_DIFFUSION')
BRANCH   = 'week-4'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',REPO_URL,REPO_DIR], check=True)
else:
    subprocess.run(['git','-C',REPO_DIR,'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',REPO_DIR,'reset','--hard',f'origin/{BRANCH}'], check=True)
# --no-deps avoids the (harmless) RAPIDS dependency-resolver warning on Kaggle
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','pytorch-msssim'], check=True)
if PKG_ROOT not in sys.path:
    sys.path.insert(0, PKG_ROOT)
print('import root:', PKG_ROOT)

In [ ]:
import torch
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  [{i}] {torch.cuda.get_device_name(i)}  '
          f'{torch.cuda.get_device_properties(i).total_memory/1e9:.0f} GB')

## 2. Imports, device, data


In [ ]:
import glob, time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src.models.noise_scheduler import NoiseScheduler
from src.models.diffusion_unet import default_diffusion_config, DiffusionUNet
from src.training.diffusion import DenoisingDiffusion
from src.data.dataset import create_dataloaders

device = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('device:', device)

In [ ]:
def find_data():
    hits = glob.glob('/kaggle/input/**/dirty.npy', recursive=True)
    if hits: return os.path.dirname(hits[0])
    if os.path.exists(os.path.join(PKG_ROOT, 'data', 'dirty.npy')): return os.path.join(PKG_ROOT, 'data')
    raise FileNotFoundError('Upload dirty.npy/clean.npy as a Kaggle Dataset and *Add Input*.')

DATA_DIR = find_data(); print('data dir:', DATA_DIR)
dirty = np.load(os.path.join(DATA_DIR,'dirty.npy')).astype(np.float32)
clean = np.load(os.path.join(DATA_DIR,'clean.npy')).astype(np.float32)
print('dirty', dirty.shape, '| clean', clean.shape, '| range [%.3f, %.3f]' % (dirty.min(), dirty.max()))

idx = np.arange(len(dirty))
train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=SEED, shuffle=True)
print('train', len(train_idx), '| val', len(val_idx))

## 3. Noise scheduler + forward diffusion q(xₜ|x₀)  (Day 1.2)

The scheduler (`src/models/noise_scheduler.py`, PR #22) implements linear and cosine β-schedules.
Forward process: `xₜ = √ᾱₜ·x₀ + √(1−ᾱₜ)·ε`. Below: one clean 128×128 patch noised at
**t = 0, 250, 500, 750, 999**.


In [ ]:
RES = 128
sched_lin = NoiseScheduler(timesteps=1000, beta_schedule='linear')

pi = int(val_idx[0])
r0 = (dirty.shape[1]-RES)//2; c0 = (dirty.shape[2]-RES)//2
cp = clean[pi, r0:r0+RES, c0:c0+RES].astype(np.float32)
lo, hi = cp.min(), cp.max(); cp = (cp-lo)/(hi-lo) if hi > lo else cp
x0 = torch.from_numpy(cp[None,None]).float()

steps = [0, 250, 500, 750, 999]
fig, ax = plt.subplots(1, len(steps), figsize=(16, 3.3))
for k, t in enumerate(steps):
    xt, _ = sched_lin.q_sample(x0, torch.tensor([t]))
    ax[k].imshow(xt[0,0].numpy(), cmap='inferno'); ax[k].axis('off'); ax[k].set_title(f't={t}')
plt.suptitle('Forward diffusion q(xₜ|x₀) — progressive noising', fontweight='bold')
plt.tight_layout(); plt.savefig('/kaggle/working/forward_diffusion.png', dpi=130); plt.show()

### Linear vs cosine β-schedule


In [ ]:
sched_cos = NoiseScheduler(timesteps=1000, beta_schedule='cosine')
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(sched_lin.betas.numpy(), label='linear'); ax[0].plot(sched_cos.betas.numpy(), label='cosine')
ax[0].set_title('βₜ'); ax[0].set_xlabel('t'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(sched_lin.alphas_cumprod.numpy(), label='linear'); ax[1].plot(sched_cos.alphas_cumprod.numpy(), label='cosine')
ax[1].set_title('ᾱₜ (cumulative signal)'); ax[1].set_xlabel('t'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.suptitle('Beta schedules', fontweight='bold'); plt.tight_layout(); plt.show()

BETA_SCHEDULE = 'linear'   # set 'cosine' to try the cosine schedule

## 4. Conditional U-Net — scaled config (Day 2)

6-level `ch=128` DDPM backbone (the plan's 51M-style config; ~110M params at this depth). Conditional:
input is `[dirty, noisy_clean]` (2 ch) → predicts noise (1 ch). Verify the forward shape
**(B, 2, 128, 128) → (B, 1, 128, 128)**.


In [ ]:
def make_cfg(res):
    c = default_diffusion_config(image_size=res)
    c.model.ch = 128
    c.model.ch_mult = [1, 1, 2, 2, 4, 4]    # 6 levels (attn @16)
    c.model.attn_resolutions = [16]
    c.diffusion.beta_schedule = BETA_SCHEDULE
    return c

cfg = make_cfg(RES)
_m = DiffusionUNet(cfg).to(device)
with torch.no_grad():
    _o = _m(torch.randn(2,2,RES,RES,device=device), torch.randint(0,1000,(2,),device=device))
print('params:', f'{sum(p.numel() for p in _m.parameters()):,}', '| forward', tuple(_o.shape))
assert _o.shape == (2,1,RES,RES)
del _m, _o; torch.cuda.empty_cache()

## 5. Data loaders + runner (Day 3)
2-channel `[dirty, clean]` 128×128 patches; `DenoisingDiffusion` auto-wraps `nn.DataParallel`, runs EMA + Adam lr=2e-5.


In [ ]:
BATCH_IMAGES = 4    # images/batch  (effective patches = BATCH_IMAGES * PATCH_N)
PATCH_N      = 4    # 128px patches per image  -> effective batch 16

def build_runner():
    \"\"\"Build at 128px; on CUDA OOM, halve batch then fall back to 64px.\"\"\"
    global BATCH_IMAGES
    res = RES
    for _ in range(5):
        run = DenoisingDiffusion(config=make_cfg(res), device=device, lr=2e-5,
                                 checkpoint_path='/kaggle/working/diffusion_scaled_best.pth.tar')
        try:
            tl, _ = create_dataloaders(dirty_train=dirty[train_idx], clean_train=clean[train_idx],
                                       batch_size=BATCH_IMAGES, num_workers=0,
                                       parse_patches=True, patch_size=res, n_patches=PATCH_N)
            xb, _ = next(iter(tl)); xb = xb.flatten(0,1) if xb.ndim==5 else xb
            xb = (2*xb-1).to(device)
            import src.training.diffusion as dm
            e = torch.randn_like(xb[:,1:]); t = torch.randint(0, run.num_timesteps, (xb.size(0),), device=device)
            dm.noise_estimation_loss(run.model, xb, t, e, run.betas).backward(); run.optimizer.zero_grad()
            print(f'[OK] {res}px, batch_img={BATCH_IMAGES}, '
                  f'{sum(p.numel() for p in run._core.parameters()):,} params, '
                  f'GPUs={run.num_gpus}{" DataParallel" if run.data_parallel else ""}')
            return run, res
        except RuntimeError as ex:
            if 'out of memory' not in str(ex).lower(): raise
            torch.cuda.empty_cache(); del run
            if BATCH_IMAGES > 1: BATCH_IMAGES //= 2; print(f'[OOM] retry batch_img={BATCH_IMAGES}')
            elif res > 64:       res = 64;            print(f'[OOM] fall back to {res}px')
            else: raise
    raise RuntimeError('could not fit the model')

diffusion, RES_USED = build_runner()
train_loader, val_loader = create_dataloaders(
    dirty_train=dirty[train_idx], clean_train=clean[train_idx],
    dirty_val=dirty[val_idx],     clean_val=clean[val_idx],
    batch_size=BATCH_IMAGES, num_workers=2,
    parse_patches=True, patch_size=RES_USED, n_patches=PATCH_N)
print('training at', RES_USED, 'px | train batches', len(train_loader))

## 6. Train (Day 3) — loss logged every 10 steps

Diffusion models need **many** iterations to converge — far more than the autoencoders. Start with a
modest `EPOCHS` to confirm the pipeline end-to-end, then raise it for a real run (T4×2 is fast).


In [ ]:
EPOCHS = 30   # raise (e.g. 100-300) once you've confirmed the pipeline runs

res = diffusion.train(train_loader, val_loader, n_epochs=EPOCHS, log_every_step=10)
print('\nbest val noise-loss:', round(res['best_val_loss'], 4))

## 7. Loss curve (deliverable)


In [ ]:
tr, va = res['train_losses'], res['val_losses']
plt.figure(figsize=(8,5))
plt.plot(range(1,len(tr)+1), tr, marker='o', ms=3, label='train')
plt.plot(range(1,len(va)+1), va, marker='s', ms=3, label='val')
plt.xlabel('epoch'); plt.ylabel('noise-estimation loss')
plt.title(f'Scaled conditional DDPM (ch=128, 6-level, {RES_USED}px)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('/kaggle/working/diffusion_scaled_loss.png', dpi=130); plt.show()

## 8. DDIM sampling (50 steps) + first samples + metrics (Day 4)


In [ ]:
diffusion.load_checkpoint('/kaggle/working/diffusion_scaled_best.pth.tar')
m = diffusion.evaluate(val_loader, sampling_timesteps=50, max_batches=8, use_ema=True)
print(f"DDPM ({RES_USED}px)  n={m['n']}  PSNR={m['psnr']:.4f} dB  SSIM={m['ssim']:.4f}  MSE={m['mse']:.6f}")
print('(preliminary — diffusion needs many more epochs to converge)')

In [ ]:
xb, _ = next(iter(val_loader)); xb = xb.flatten(0,1) if xb.ndim==5 else xb
dp = xb[:5, 0:1].clamp(0,1); cp = xb[:5, 1:2].clamp(0,1)
pp = diffusion.sample(dp, sampling_timesteps=50, use_ema=True).cpu()
fig, ax = plt.subplots(3, 5, figsize=(15, 9))
for i,(im,nm) in enumerate([(dp,'dirty'),(pp,'DDPM'),(cp,'clean')]):
    for j in range(5): ax[i,j].imshow(im[j,0], cmap='inferno'); ax[i,j].axis('off')
    ax[i,0].set_title(nm, loc='left', fontsize=12)
plt.suptitle(f'First sample outputs — scaled DDPM @ {RES_USED}px', fontweight='bold')
plt.tight_layout(); plt.savefig('/kaggle/working/diffusion_scaled_samples.png', dpi=130); plt.show()

## 9. Outputs (Friday demo materials)

Saved to `/kaggle/working/` (Output tab):
- `forward_diffusion.png` — q(xₜ|x₀) visualization
- `diffusion_scaled_loss.png` — training loss curve
- `diffusion_scaled_samples.png` — dirty → DDPM → clean
- `diffusion_scaled_best.pth.tar` — best checkpoint (model + EMA + optimizer + config)

**Talking points:** migrated to Kaggle T4×2; conditional DDPM scaled to the 6-level ch=128 backbone
(~110M params) at 128×128; training is preliminary (diffusion needs far more steps than the
autoencoders); full benchmark next week.
